In [1]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import os
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd
import seaborn as sns
import pickle as pkl
from tqdm import tqdm
import csv
from joblib import Parallel, delayed
from collections import Counter
from itertools import product
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import SelectKBest, chi2, f_regression

In [2]:
asm_file_size = pd.read_csv("asm_file_sizes.csv")
unigram_asm = pd.read_csv("output/final_asm_features_with_class.csv")

In [7]:
unigram_asm_feature__with_size=pd.merge(asm_file_size, unigram_asm.drop(columns=["Class"]),on='ID', how='left')

unigram_asm_feature__with_size.to_csv("featurization/unigram_asm_feature__with_size", index=False)

unigram_asm_feature__with_size.head()

,ID,size,Class,HEADER:,.text:,.Pav:,.idata:,.data:,.bss:,.rdata:,...,eax,ebx,ecx,edi,ebp,esp,eip,.dll,std::,:dword
0,01IsoiSMh5gxyDYTl4CB,13.999378,2,0,109939,0,616,24568,0,26405,...,1446,260,1090,391,905,420,0,24,22,227
1,01jsnpXSAlgw6aPeDxrU,8.507785,9,18,68883,0,304,662,0,1093,...,903,5,547,5,451,56,0,27,0,117
2,02JqQ7H3yEoD8viYWlmS,17.073298,2,0,129362,0,644,24994,0,24509,...,480,147,353,168,375,63,0,22,0,236
3,02K5GMYITj7bBoAisEmD,5.488952,2,0,93532,0,503,6551,0,21273,...,19726,12761,17438,13613,6078,301,0,44,17,209
4,02MRILoE6rNhmt7FUi45,83.639698,2,0,5372,0,606,4329,0,2349368,...,835,263,665,291,664,256,0,19,16,203


In [8]:
unigram_asm_feature__with_size = pd.read_csv(
    "featurization/unigram_asm_feature__with_size"
).drop(["Class", "rtn", ".BSS:", ".CODE"], axis=1)
unigram_asm_feature__with_size

,ID,size,HEADER:,.text:,.Pav:,.idata:,.data:,.bss:,.rdata:,.edata:,...,eax,ebx,ecx,edi,ebp,esp,eip,.dll,std::,:dword
0,01IsoiSMh5gxyDYTl4CB,13.999378,0,109939,0,616,24568,0,26405,0,...,1446,260,1090,391,905,420,0,24,22,227
1,01jsnpXSAlgw6aPeDxrU,8.507785,18,68883,0,304,662,0,1093,0,...,903,5,547,5,451,56,0,27,0,117
2,02JqQ7H3yEoD8viYWlmS,17.073298,0,129362,0,644,24994,0,24509,0,...,480,147,353,168,375,63,0,22,0,236
3,02K5GMYITj7bBoAisEmD,5.488952,0,93532,0,503,6551,0,21273,0,...,19726,12761,17438,13613,6078,301,0,44,17,209
4,02MRILoE6rNhmt7FUi45,83.639698,0,5372,0,606,4329,0,2349368,0,...,835,263,665,291,664,256,0,19,16,203
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10863,Lf5U8qioZ9yOkjrGT37A,3.020143,17,564,0,107,67600,0,308,0,...,36,20,16,11,12,20,0,4,0,41
10864,LivEmF2ytpsDexIVWuYR,3.439602,17,3403,0,3,27848,0,55313,3,...,279,189,152,122,93,135,0,0,0,0
10865,lkqEXK4NrYSseRTt0Gb3,0.629995,17,2692,0,0,10919,0,0,0,...,182,96,69,72,33,134,0,0,0,0
10866,loIP1tiwELF9YNZQjSUO,11.269457,17,631,0,109,264208,0,317,0,...,43,29,16,8,15,17,0,4,0,42
